# 01 - Exploratory analysis of the input data (before any correction)

**Purpose**: see what each dataset contains before correcting anything: grid, period, calendar, units,
gaps, outliers, dry days, order of magnitude of the GCM / ERA5 / RCM differences.

Nothing is modified. Paths come from notebook 00 (`config_<profile>.json`), never from here.

To stay fast, statistics are computed on a **sample** (one year out of `SAMPLE_STEP`,
and one pixel out of `SPACE_STEP` for fine grids). Set `SAMPLE_STEP = 1` to read everything.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent, Path.home() / "downscaling"):   # ba_utils.py: notebook folder, parent folder, or the repository
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']
DOMAIN = bu.rcm_domain(cfg)   # RCM domain for regional diagnostics (None = global until the RCM path is filled in)

SAMPLE_STEP = 5      # one year out of 5
SPACE_STEP = 4       # one pixel out of 4 (0.25 deg grids) for the global statistics
PLOTS = bu.explo_dir("01_exploration_data")   # ~/001_exploration/outputs/01_exploration_data (private)

## 1. Detailed inventory

In [ ]:
def sample(da, space=1):
    yrs = np.unique(da.time.dt.year.values)[::SAMPLE_STEP]
    idx = np.where(np.isin(da.time.dt.year.values, yrs))[0]
    return da.isel(time=idx, lat=slice(None, None, space), lon=slice(None, None, space))


avail, rows = {}, []
for name in cfg["datasets"]:
    for var in VARS:
        if not bu.dataset_available(cfg, name, var):
            rows.append({"dataset": name, "var": var, "status": "missing / to fill in (notebook 00)"})
            continue
        da = bu.read_raw(cfg, name, var)
        avail[(name, var)] = da
        t = da.indexes["time"]
        years = np.unique(da.time.dt.year.values)
        space = SPACE_STEP if da.sizes["lat"] > 400 else 1
        s = sample(da, space).compute()
        rows.append({
            "dataset": name, "var": var, "status": "ok",
            "period": f"{years[0]}-{years[-1]}", "n_days": da.sizes["time"],
            "expected_days(noleap)": len(years) * 365,
            "duplicated_times": int(pd.Index(t).duplicated().sum()),
            "original_calendar": da.attrs["orig_calendar"],
            "original_units": da.attrs["orig_units"],
            "nlat x nlon": f"{da.sizes['lat']} x {da.sizes['lon']}",
            "lat_step": round(float(np.median(np.diff(da.lat))), 3),
            "lon_step": round(float(np.median(np.diff(da.lon))), 3),
            "lat_descending_originally": bool(da.attrs["orig_lat_desc"]),
            "lon_0_360_originally": bool(da.attrs["orig_lon_0360"]),
            "mean": round(float(s.mean()), 3), "min": round(float(s.min()), 3), "max": round(float(s.max()), 3),
            "frac_NaN": round(float(s.isnull().mean()), 5),
            "frac_dry_days(<0.1mm)": round(float((s < bu.THRESH_PR).mean()), 3) if var == "pr" else None,
        })
inv = pd.DataFrame(rows)
inv.to_csv(PLOTS / "detailed_inventory.csv", index=False)
pd.set_option("display.width", 250, "display.max_columns", 30)
inv

## 2. Automatic alerts

In [ ]:
alerts = []
ok = inv[inv["status"] == "ok"]
for _, r in ok.iterrows():
    k = f"{r['dataset']}/{r['var']}"
    if r["n_days"] != r["expected_days(noleap)"]:
        alerts.append(f"{k}: {r['n_days']} days for {r['expected_days(noleap)']} expected (gaps or partial years)")
    if r["duplicated_times"]:
        alerts.append(f"{k}: {r['duplicated_times']} duplicated dates")
    if r["frac_NaN"] > 0:
        alerts.append(f"{k}: {100*r['frac_NaN']:.2f} % NaN in the sample")
    if r["var"] == "pr" and r["min"] < 0:
        alerts.append(f"{k}: negative precipitation (min = {r['min']}) -> clip to 0 before bias adjustment")
    if r["var"] != "pr" and not (-90 < r["min"] and r["max"] < 70):
        alerts.append(f"{k}: temperature outside plausible range ({r['min']} .. {r['max']} degC)")
    if r["original_calendar"] not in ("noleap", "365_day"):
        alerts.append(f"{k}: original calendar '{r['original_calendar']}' -> converted to noleap (29 February removed)")
    if r["lat_descending_originally"] or r["lon_0_360_originally"]:
        alerts.append(f"{k}: descending lat / lon 0-360 originally -> normalised (ascending lat, lon -180..180)")
print("\n".join(alerts) if alerts else "No alert.")
print("\nCalendars:", sorted(ok["original_calendar"].unique()))

## 3. Mean seasonal cycle over the RCM domain (sample; global until the RCM path is filled in)

The curves should have the same shape. Level differences (GCM / RCM vs ERA5) are what bias adjustment will address.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, var in zip(axes, VARS):
    for (name, v), da in avail.items():
        if v != var:
            continue
        sub = bu.crop_bbox(da, DOMAIN)
        sub = sample(sub, SPACE_STEP if sub.sizes["lat"] > 400 else 1)
        m = bu.area_mean(sub).groupby("time.month").mean().compute()
        ax.plot(m.month, m, marker="o", ms=3, label=name)
    ax.set_title(f"{var} - domain mean"); ax.set_xlabel("month")
    ax.set_ylabel("mm/day" if var == "pr" else "degC")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(PLOTS / "seasonal_cycle_domain.png", dpi=120); plt.show()

## 4. Maps of the mean and of dry days: ERA5 vs EC-Earth3 historical (native grids)

In [ ]:
def mean_map(da, space):
    return sample(da, space).mean("time").compute()

pairs = [("era5", "gcm_historical")]
for var in ["pr", "tasmax"]:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, name in zip(axes, pairs[0]):
        if (name, var) not in avail:
            ax.set_title(f"{name}: missing"); continue
        da = avail[(name, var)]
        m = mean_map(da, SPACE_STEP if da.sizes["lat"] > 400 else 1)
        im = ax.pcolormesh(m.lon, m.lat, m, shading="auto"); plt.colorbar(im, ax=ax, shrink=0.8)
        ax.set_title(f"{name} - mean {var} ({'mm/day' if var == 'pr' else 'degC'})")
    plt.tight_layout(); plt.savefig(PLOTS / f"map_mean_{var}.png", dpi=110); plt.show()

if ("era5", "pr") in avail and ("gcm_historical", "pr") in avail:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, name in zip(axes, pairs[0]):
        da = avail[(name, "pr")]
        s = sample(da, SPACE_STEP if da.sizes["lat"] > 400 else 1)
        f = (s < bu.THRESH_PR).mean("time").compute()
        im = ax.pcolormesh(f.lon, f.lat, f, shading="auto", vmin=0, vmax=1); plt.colorbar(im, ax=ax, shrink=0.8)
        ax.set_title(f"{name} - dry-day fraction (< {bu.THRESH_PR} mm)")
    plt.tight_layout(); plt.savefig(PLOTS / "map_dry_days.png", dpi=110); plt.show()

## 5. Pooled distributions over the RCM domain (quantiles)

In [ ]:
rows = []
for (name, var), da in avail.items():
    sub = bu.crop_bbox(da, DOMAIN)
    sub = sample(sub, SPACE_STEP if sub.sizes["lat"] > 400 else 1)
    v = sub.values.ravel(); v = v[np.isfinite(v)]
    if v.size == 0:
        continue
    q = np.percentile(v, [1, 5, 25, 50, 75, 95, 99, 99.9])
    r = dict(zip(["p1", "p5", "p25", "p50", "p75", "p95", "p99", "p99.9"], q.round(2)))
    r.update({"dataset": name, "var": var, "n": v.size})
    if var == "pr":
        r["dry_frac"] = round(float((v < bu.THRESH_PR).mean()), 3)
        w = v[v >= bu.THRESH_PR]
        r["p95_wet_days"] = round(float(np.percentile(w, 95)), 2) if w.size else None
    rows.append(r)
dist = pd.DataFrame(rows).set_index(["var", "dataset"]).sort_index()
dist.to_csv(PLOTS / "quantiles_domain.csv")
dist

## 6. tasmax / tasmin consistency

In [ ]:
rows = []
for name in cfg["datasets"]:
    if (name, "tasmax") in avail and (name, "tasmin") in avail:
        tx, tn = sample(avail[(name, "tasmax")], SPACE_STEP if avail[(name, "tasmax")].sizes["lat"] > 400 else 1), \
                 sample(avail[(name, "tasmin")], SPACE_STEP if avail[(name, "tasmin")].sizes["lat"] > 400 else 1)
        dtr = (tx - tn).compute()
        rows.append({"dataset": name, "mean DTR (degC)": round(float(dtr.mean()), 2),
                     "frac DTR <= 0": round(float((dtr <= 0).mean()), 5)})
pd.DataFrame(rows)

## 7. To keep in mind before notebook 02

Read the alert list (section 2): everything reported (units, calendar, longitudes, NaN, gaps) must be understood
before going further. Units and calendar are already converted automatically by `ba_utils.read_raw`
(mm/day, degC, `noleap` calendar); check that the conversion is the one you expect.